# 4. Training a Phase Reconstructor with `Trainer`

Part 4 (final) of the tutorial series (`01_Dataset.ipynb`, `02_WFSAndPreprocessing.ipynb`, `03_DeformableMirrorAndClosedLoop.ipynb`). Notebook 3 closed the AO loop with an oracle reconstructor that cheated by using the true residual phase directly. This notebook replaces that oracle with a real reconstructor — a CNN that only ever sees the WFS's detector frame, exactly like a real system would — and trains it end-to-end through the whole differentiable pipeline (WFS → DM) with `AI4AO.Trainer.Trainer` (see `AI4AO/Trainer.py`).

## Configuration, dataset, WFS, DM and frame preprocessor

Same setup as notebook 3, now with the WFS and DM frozen (`.eval()`) since only the reconstructor network's weights are trained here.

### The `TrainParams` fields

The last of the five `wfs_params_exp.py` dicts, used only from this notebook on:

- `lro`: learning rate for the optical/mask parameters, when `OptimizeMask = True` (the WFS mask itself is trained, not just the reconstructor). Unused in this notebook, since only the reconstructor is optimized below.
- `lrn`: learning rate for the reconstructor network's optimizer (`optimizer_n` below).
- `TrainRunNb`: default number of closed-loop training steps to run (`trainer.train`).
- `TestRunNb`: number of steps used for evaluation rollouts.
- `OptimizeMask`: whether the WFS mask's own parameters are included among the trainable parameters — left `False` here, so `wfs` stays frozen and only `phaseReconstructor` is trained.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow_multiple
from AI4AO.LossFunctions import LogResidualVarianceLoss, Physics_loss

device = 'cuda'  # set to "cpu" if CUDA is not available

paramfile = 'wfs_params_exp.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']
DMParams = Config.fromfile(paramfile)['DMParams']

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

wfs = PyramidWFS(WFSParams, device)
wfs.eval()

dm = DeformableMirror(WFSParams, DMParams, device)
dm.offset_to_fit_number_of_actuators = 0.1
dm.eval()


framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

M2C = torch.eye(DMParams["Nmodes"], device=device)
z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim=-2))

## Reconstructor architecture

`PWFSNet` maps the WFS's 4 preprocessed pyramid pupil images to a vector of `Nmodes` actuator coefficients: a small convolutional encoder (grouped convolutions in the stem, so each pupil is processed independently at first) followed by a linear head.

In [ ]:
class PWFSNet(nn.Module):
    def __init__(self, DMParams):
        super().__init__()

        Nmodes = DMParams["Nmodes"]

        self.stem = nn.Sequential(
            # Process each pupil independently
            nn.Conv2d(4, 32, kernel_size=11, padding=5, groups=4),
            nn.GELU(),

            nn.Conv2d(32, 64, kernel_size=7, padding=3, groups=4),
            nn.GELU(),

            nn.MaxPool2d(2),      # 42 -> 21
        )

        self.encoder = nn.Sequential(
            nn.Conv2d(64, 64, 5, padding=2),
            nn.GELU(),

            nn.MaxPool2d(2),      # 21 -> 10

            nn.Conv2d(64, 128, 3, padding=1),
            nn.GELU(),

            nn.MaxPool2d(2),      # 10 -> 5

            nn.Conv2d(128, 256, 3, padding=1),
            nn.GELU(),

            nn.MaxPool2d(2),      # 10 -> 5

            nn.Conv2d(256, 512, 2, padding=1),
            nn.GELU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, Nmodes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.encoder(x)
        return self.head(x)

## Optimizer and loss

We optimize only the reconstructor's parameters, with AdamW. `LogResidualVarianceLoss` computes `ln(var(residual phase over the pupil))` — a physically meaningful quantity (tied to residual RMS/Strehl ratio), not just an abstract regression loss on the coefficients. `Physics_loss` propagates the reconstructed phase through the forward WFS model and compares it to the real WFS frame. This can work as a unsupervised learning method. You can combine loss function as a weighted sum to benefit from each.

In [ ]:
phaseReconstructor = PWFSNet(DMParams).to(device=device)

total_params = sum(p.numel() for p in phaseReconstructor.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

lrn = TrainParams['lrn']
optimizer_n = torch.optim.AdamW(phaseReconstructor.parameters(), lrn, fused=True)

loss = LogResidualVarianceLoss(dataset.pupil) + Physics_loss(wfs=wfs)*100

## The Trainer

`AI4AO.Trainer.Trainer` bundles the WFS, DM, frame preprocessor, modal basis (`M2C`), reconstructor, dataset, loss and optimizer, and implements the closed-loop training step: draw a phase screen, propagate the residual through the WFS, reconstruct it, backpropagate through the WFS/DM simulation into the network. It also exposes `save_checkpoint`/`load_checkpoint`, and the `evaluate`/`plot_losses` helpers used further down.

`CHECKPOINT_PATH` points at a tutorial-series-specific location under `Data/` — there's no `Data/<Instrument>/` folder for this synthetic setup, since it isn't tied to a real instrument.

In [ ]:
CHECKPOINT_PATH = "../../Data/Tutorials/ReconstructorCNN.pth"

trainer = Trainer(
    wfs=wfs,
    framePreprocessor=framePreprocessor,
    dm=dm,
    M2C=M2C,
    phaseReconstructor=phaseReconstructor,
    dataset=dataset,
    loss=loss,
    optimizer=optimizer_n,
)

trainer.load_checkpoint(CHECKPOINT_PATH, load_optimizer=False)

## Training

`trainer.train(training_steps, closed_loop_iterations)` runs `training_steps` closed-loop optimizer updates. `closed_loop_iterations` sets how many AO-loop steps are simulated — and backpropagated through — per optimizer update; with more than 1, the reconstructor is trained to perform well *given* its own previous corrections, rather than only on independent open-loop frames.

It returns two per-step loss trackers: `loss_tracker`, the network's actual training loss, and `loss_tracker_ideal` — the loss from the same oracle reconstruction as notebook 3 (`z_inv` applied to the true residual phase) instead of the network's prediction. That's the lower bound the network is chasing.

In [ ]:
TrainRunNb = 1000
num_iterations = 1  # closed-loop iterations backpropagated through per optimizer step

loss_tracker, loss_tracker_ideal = trainer.train(TrainRunNb, num_iterations)

`trainer.plot_losses` smooths and plots both trackers together. The gap between the training loss and the oracle's ideal-loss lower bound indicates how much reconstruction performance is still on the table for the network to gain, versus how much is fundamental to the WFS/DM themselves.

In [ ]:
trainer.plot_losses(loss_tracker, loss_tracker_ideal)

## Saving

Persist the trained reconstructor and optimizer state via `trainer.save_checkpoint`.

In [ ]:
trainer.save_checkpoint(CHECKPOINT_PATH)

## Visualizing the trained closed loop

`trainer.evaluate()` runs a no-grad closed-loop rollout (reconstructor in `.eval()` mode, no pupil noise injected) and returns an `EvaluationResult` holding the phase, pupil, reconstructed phase, residual phase and WFS frames at every simulated step, ready to animate — the same rollout structure as notebook 3, except the correction now comes from the trained network reading the WFS frame instead of the oracle reading the true phase.

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

n_frames = 100
result = trainer.evaluate(n_steps=n_frames, dataset=dataset)

fig, axes = imshow_multiple(
    [result.phase[0], result.residual_phase[0], result.wfs_frames[0], torch.sqrt(result.psfs[0])],
    same_scale=True,
    titles=["Input phase", "Residual phase", "WFS frame", "PSF"],
    max_channel_number = 9
)


def update(i):
    imshow_multiple(
        [result.phase[i], result.residual_phase[i], result.wfs_frames[i], torch.sqrt(result.psfs[i])],
        fig=fig, axes=axes, 
        same_scale=True,
        max_channel_number = 9
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())

### With scintillation

Same rollout, but on a fresh dataset with `AtmosParams['Scintillation'] = True` — amplitude fluctuations from angular-spectrum propagation are added on top of the phase screens, which the reconstructor was not trained to handle. Useful to see how it degrades outside its training distribution.

In [ ]:
scint_AtmosParams = AtmosParams.copy()
scint_AtmosParams['Scintillation'] = True
scint_dataset = PhaseDataset(WFSParams, scint_AtmosParams, LoopParams, DMParams, device)
scint_dataset.generateClosedLoop = True

result = trainer.evaluate(n_steps=n_frames, dataset=scint_dataset)

fig, axes = imshow_multiple(
    [result.pupil[0] * wfs.pupil, result.phase[0], result.residual_phase[0], result.wfs_frames[0], torch.sqrt(result.psfs[0])],
    same_scale=True,
    titles=["Pupil amplitude", "Input phase", "Residual phase", "WFS frame", "PSF"],
    max_channel_number = 9
)


def update(i):
    imshow_multiple(
        [result.pupil[i] * wfs.pupil, result.phase[i], result.residual_phase[i], result.wfs_frames[i], torch.sqrt(result.psfs[i])],
        fig=fig, axes=axes, 
        same_scale=True,
        max_channel_number = 9
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=True)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())